# Lenguajes y Autómatas II (SCD-1016)
## Sesión 02: Memoria Dinámica, Nodos y Estructura de Repositorio
**Docente:** Mtro. Iván Márquez (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/lenguajes-automatas-2`

---
### Objetivos del Laboratorio
1. Manipular asignación dinámica en memoria (`malloc`, `free`) para crear nodos dinámicos.
2. Identificar fugas de memoria (*Memory Leaks*) y apuntadores colgantes (*Dangling Pointers*).
3. Llenar y registrar el archivo `auditorias/auditoria_sesion02.md` en tu Fork personal.

---
### Fase 1: El Artesano (15 min - Sin IA)
**Reto: Construcción de una Lista Enlazada de Tokens**

En un compilador, los tokens y nodos del AST se crean dinámicamente en el Heap. Completa la función `crear_token()` y `liberar_lista()` asegurando que no existan fugas de memoria.

In [ ]:
%%writefile tokens_memoria.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

typedef struct Token {
    char lexema[32];
    int tipo;
    struct Token *siguiente;
} Token;

// TODO: Reserva memoria con malloc, inicializa los campos y retorna el apuntador
Token* crear_token(const char *lexema, int tipo) {
    Token *nuevo = (Token*)malloc(sizeof(Token));
    if (nuevo == NULL) {
        fprintf(stderr, "Error de asignacion de memoria\n");
        exit(1);
    }
    strncpy(nuevo->lexema, lexema, sizeof(nuevo->lexema) - 1);
    nuevo->lexema[sizeof(nuevo->lexema) - 1] = '\0';
    nuevo->tipo = tipo;
    nuevo->siguiente = NULL;
    return nuevo;
}

// TODO: Recorre la lista y libera cada nodo sin generar punteros colgantes
void liberar_lista(Token *cabeza) {
    Token *actual = cabeza;
    while (actual != NULL) {
        Token *aux = actual->siguiente;
        free(actual);
        actual = aux;
    }
}

int main() {
    printf("[COMPILADOR] Creando secuencia de tokens...\n");
    
    Token *cabeza = crear_token("int", 101);
    cabeza->siguiente = crear_token("identificador_x", 202);
    cabeza->siguiente->siguiente = crear_token(";", 303);
    
    Token *temp = cabeza;
    while (temp != NULL) {
        printf("Token: %-18s | Tipo: %d | Mem: %p\n", temp->lexema, temp->tipo, (void*)temp);
        temp = temp->siguiente;
    }
    
    liberar_lista(cabeza);
    printf("[MEMORIA] Lista liberada correctamente.\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra tokens_memoria.c -o tokens_memoria && ./tokens_memoria

---
### Track Alternativo en Rust (Propiedad y Borrow Checker)

In [ ]:
%%writefile tokens_memoria.rs
#[derive(Debug)]
struct Token {
    lexema: String,
    tipo: i32,
    siguiente: Option<Box<Token>>,
}

fn main() {
    let nodo3 = Token { lexema: ";".to_string(), tipo: 303, siguiente: None };
    let nodo2 = Token { lexema: "identificador_x".to_string(), tipo: 202, siguiente: Some(Box::new(nodo3)) };
    let cabeza = Token { lexema: "int".to_string(), tipo: 101, siguiente: Some(Box::new(nodo2)) };
    
    println!("[RUST MEM] Lista enlazada autogestionada (RAII):\n{:#?}", cabeza);
}

In [ ]:
!rustc tokens_memoria.rs -o tokens_rust && ./tokens_rust